# Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[0] ## Path().resolve() - notebook location and parents[0] goes from notebooks/ to project_root/
print(project_root)
sys.path.append(str(project_root))

In [ ]:
import logging

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.basemap import Basemap
import xarray as xr
from numpy.lib.stride_tricks import sliding_window_view

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import json, pickle
import glob, os
import joblib
from collections import defaultdict

In [ ]:
# from utils.utils import sample_random
import utils.utils as util_functions
from IPython.display import display, clear_output

In [ ]:
import torch
from torch import nn, utils
from torch.utils.data import TensorDataset, DataLoader, Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

In [ ]:
from models.lstm import LSTMAttnHeteroRegressor
from models.tcn import TCNAttenHeteroRegressor

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Load data

In [ ]:
sim_num = 1
list_of_files = glob.glob(f"DATA-PATH")
output_path = '_files_for_analysis_'

# Build tensors

In [ ]:
%%time

save_dir = f"{output_path}/{sim_num}"

with open(os.path.join(save_dir, "1958_1987_8f_12m_feature_dict.pkl"), "rb") as f:
    feature_dict_train = pickle.load(f)

with open(os.path.join(save_dir, "1958_1987_8f_12m_target_dict.pkl"), "rb") as f:
    target_dict_train = pickle.load(f)

with open(os.path.join(save_dir, "1988_1989_8f_12m_feature_dict.pkl"), "rb") as f:
    feature_dict_cal = pickle.load(f)

with open(os.path.join(save_dir, "1988_1989_8f_12m_target_dict.pkl"), "rb") as f:
    target_dict_cal = pickle.load(f)

with open(os.path.join(save_dir, "1990_1999_8f_12m_feature_dict.pkl"), "rb") as f:
    feature_dict_val = pickle.load(f)

with open(os.path.join(save_dir, "1990_1999_8f_12m_target_dict.pkl"), "rb") as f:
    target_dict_val = pickle.load(f)

In [ ]:
%%time
def build_xy_from_dicts(feature_dict, target_dict, lookback=6, dtype=np.float32,
                        start_year=1958):
    """
    Returns:
      X_all:    (N_samples, lookback, 8)
      y_all:    (N_samples, 1)
      coord_all:(N_samples, 2)   lat, lon
      time_all: (N_samples, 2)   year, month
    """
    X_list, y_list, coord_list, time_list = [], [], [], []

    for (lat, lon), year_blocks in feature_dict.items():
        if (lat, lon) not in target_dict:
            continue

        # (T, 8)
        X_ts = np.concatenate(year_blocks, axis=0).astype(dtype, copy=False)
        # (T, 1)
        y_ts = np.concatenate(target_dict[(lat, lon)], axis=0).astype(dtype, copy=False)

        T, F = X_ts.shape
        if F != 8:
            raise ValueError(f"Expected 8 features, got {F}")
        if y_ts.shape != (T, 1):
            raise ValueError(f"Target shape mismatch: {y_ts.shape} vs {(T,1)}")

        # pad for early months
        pad = np.zeros((lookback - 1, F), dtype=dtype)
        X_pad = np.vstack([pad, X_ts])  # (T+lookback-1, 8)  (allocates)

        # windows as a view (no copy)
        X_win = sliding_window_view(X_pad, window_shape=(lookback,), axis=0)  # (T, 8, lookback)
        X_win = np.transpose(X_win, (0, 2, 1))  # (T, lookback, 8)

        y_win = y_ts  # (T, 1)

        X_list.append(X_win)
        y_list.append(y_win)

        # coords (T,2)
        coord_list.append(np.tile(np.array([[lat, lon]], dtype=dtype), (T, 1)))

        # time (T,2): year, month
        # t=0 is Jan of start_year
        t = np.arange(T, dtype=np.int32)
        years = start_year + (t // 12)
        months = 1 + (t % 12)
        time_list.append(np.stack([years, months], axis=1).astype(np.int16, copy=False))

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    coord_all = np.concatenate(coord_list, axis=0)
    time_all = np.concatenate(time_list, axis=0)

    return X_all, y_all, coord_all, time_all


X_train_np, y_train_np, coord_train_np, time_train_np = build_xy_from_dicts(
    feature_dict_train, target_dict_train, lookback=6, start_year=1958
)

X_cal_np, y_cal_np, coord_cal_np, time_cal_np = build_xy_from_dicts(
    feature_dict_cal, target_dict_cal, lookback=6, start_year=1988
)

X_val_np, y_val_np, coord_val_np, time_val_np = build_xy_from_dicts(
    feature_dict_val, target_dict_val, lookback=6, start_year=1990
)

In [ ]:
print(X_train_np.shape)
print(y_train_np.shape)
print(coord_train_np.shape)
print(time_train_np.shape) 

print(X_cal_np.shape)
print(y_cal_np.shape)
print(coord_cal_np.shape)
print(time_cal_np.shape) 

print(X_val_np.shape)
print(y_val_np.shape)
print(coord_val_np.shape)
print(time_val_np.shape)  

In [ ]:
%%time

N_train = X_train_np.shape[0]
k_train = 2000000   # or 20_000 5_000_000

rng = np.random.default_rng(42)
idx_train = rng.choice(N_train, size=k_train, replace=False)

X_sub_train = X_train_np[idx_train]   # shape (k, 6, 8)
y_sub_train = y_train_np[idx_train]


N_cal = X_cal_np.shape[0]
k_cal = 100000   # or 20_000 5_000_000

rng = np.random.default_rng(42)
idx_cal = rng.choice(N_cal, size=k_cal, replace=False)

X_sub_cal = X_cal_np[idx_cal]   # shape (k, 6, 8)
y_sub_cal = y_cal_np[idx_cal]


N_val = X_val_np.shape[0]
k_val = 500000  # or 20_000 5_000_000

rng = np.random.default_rng(42)
idx_val = rng.choice(N_val, size=k_val, replace=False)

X_sub_val = X_val_np[idx_val]   # shape (k, 6, 8)
y_sub_val = y_val_np[idx_val]

In [ ]:
X_train = torch.from_numpy(X_sub_train)
y_train = torch.from_numpy(y_sub_train)

X_cal = torch.from_numpy(X_sub_cal)
y_cal = torch.from_numpy(y_sub_cal)

X_val = torch.from_numpy(X_sub_val)
y_val = torch.from_numpy(y_sub_val)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_cal.shape, y_cal.shape)
print(X_val.shape, y_val.shape)

# Model training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# --- drop whole windows with any NaN/inf (GLOBAL)

valid_mask_train = np.isfinite(X_train).all(axis=(1, 2))

# print("Arrays dropped:", (~valid_mask).sum())
# print("Arrays kept:  ", valid_mask.sum())
# print("Sum:", (~valid_mask).sum() + valid_mask.sum())

X_clean_train = X_train[valid_mask_train]
y_clean_train = y_train[valid_mask_train]

valid_mask_cal = np.isfinite(X_cal).all(axis=(1, 2))
X_clean_cal = X_cal[valid_mask_cal]
y_clean_cal = y_cal[valid_mask_cal]

valid_mask_val = np.isfinite(X_val).all(axis=(1, 2))
X_clean_val = X_val[valid_mask_val]
y_clean_val = y_val[valid_mask_val]

In [ ]:
print("Train Windows dropped:", X_train.shape[0] - X_clean_train.shape[0])
print("Train Windows kept:  ", X_clean_train.shape[0])

print("Cal Windows dropped:", X_cal.shape[0] - X_clean_cal.shape[0])
print("Cal Windows kept:  ", X_clean_cal.shape[0])

print("Val Windows dropped:", X_val.shape[0] - X_clean_val.shape[0])
print("Val Windows kept:  ", X_clean_val.shape[0])

In [ ]:
print(X_clean_train.shape, X_clean_cal.shape, X_clean_val.shape)

In [ ]:
x_scaler = StandardScaler()
X_tr_2d = X_clean_train.reshape(-1, X_clean_train.shape[2])          # (Ntr*T, F)
X_ca_2d = X_clean_cal.reshape(-1, X_clean_cal.shape[2])
X_va_2d = X_clean_val.reshape(-1, X_clean_val.shape[2])

X_tr_scaled = x_scaler.fit_transform(X_tr_2d).reshape(X_clean_train.shape[0], X_clean_train.shape[1], X_clean_train.shape[2])
X_ca_scaled = x_scaler.transform(X_ca_2d).reshape(X_clean_cal.shape[0], X_clean_cal.shape[1], X_clean_cal.shape[2])
X_va_scaled = x_scaler.transform(X_va_2d).reshape(X_clean_val.shape[0], X_clean_val.shape[1], X_clean_val.shape[2])

# --- Target scaler (optional but recommended for stable regression)
# y_scaler = StandardScaler()
y_tr_scaled = y_clean_train #y_scaler.fit_transform(y_tr)   # (Ntr, 1)
y_ca_scaled = y_clean_cal #y_scaler.fit_transform(y_tr)   # (Ntr, 1)
y_va_scaled = y_clean_val #y_scaler.transform(y_va)

# Save scalers

joblib.dump(x_scaler, f"{output_path}/{sim_num}/x_scaler_k_{k_train}_for_cal.joblib")
# joblib.dump(y_scaler, f"{output_path}/{sim_num}/y_scaler_k_{k}.joblib")

print(f"Saved scalers to {output_path}/{sim_num}/")

In [ ]:
X_tr_scaled = X_tr_scaled.astype(np.float32)
X_ca_scaled = X_ca_scaled.astype(np.float32)
X_va_scaled = X_va_scaled.astype(np.float32)
y_tr_scaled = y_tr_scaled.detach().cpu().numpy().astype(np.float32)
y_ca_scaled = y_ca_scaled.detach().cpu().numpy().astype(np.float32)
y_va_scaled = y_va_scaled.detach().cpu().numpy().astype(np.float32)

print(type(X_tr_scaled), type(y_tr_scaled))
print(type(X_va_scaled), type(y_va_scaled))

In [ ]:
print(X_tr_scaled.shape, X_va_scaled.shape)

In [ ]:
%%time

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]



train_ds = SeqDataset(X_tr_scaled, y_tr_scaled)
cal_ds   = SeqDataset(X_ca_scaled, y_ca_scaled)
val_ds   = SeqDataset(X_va_scaled, y_va_scaled)

## For FNN!
# Xtrain_flat = X_tr_scaled.reshape(X_tr_scaled.shape[0], -1)
# Xval_flat   = X_va_scaled.reshape(X_va_scaled.shape[0], -1)
# train_ds = SeqDataset(Xtrain_flat, y_tr_scaled)
# val_ds   = SeqDataset(Xval_flat, y_va_scaled)

batch_size = 1024  # adjust!

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
calib_loader   = DataLoader(cal_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
%%time

## Metrics

@torch.no_grad()
def compute_metrics(y_pred, y_true):
    y_pred = y_pred.detach().cpu().numpy()
    y_true = y_true.detach().cpu().numpy()

    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae  = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae

@torch.no_grad()
def compute_metrics_with_scaler(y_pred_scaled, y_true_scaled, y_scaler):
    """
    y_pred_scaled, y_true_scaled: torch tensors (N, 1) on device
    Returns: rmse, mae in original units
    """
    y_pred = y_pred_scaled.detach().cpu().numpy()
    y_true = y_true_scaled.detach().cpu().numpy()

    y_pred_orig = y_scaler.inverse_transform(y_pred)
    y_true_orig = y_scaler.inverse_transform(y_true)

    rmse = float(np.sqrt(np.mean((y_pred_orig - y_true_orig) ** 2)))
    mae  = float(np.mean(np.abs(y_pred_orig - y_true_orig)))
    return rmse, mae


In [ ]:
import math

def heteroscedastic_gaussian_nll(mu, logvar, y):
    inv_var = torch.exp(-logvar)
    # return 0.5 * (logvar + (y - mu) ** 2 * inv_var).mean()
    return 0.5 * (math.log(2*math.pi) + logvar + (y - mu)**2 * inv_var).mean()

In [ ]:
@torch.no_grad()
def conformal_calibrate_q(model, calib_loader, device, alpha=0.05, eps=1e-6):
    """
    Returns q_hat so that intervals [mu - q_hat*sigma, mu + q_hat*sigma]
    have ~ (1-alpha) coverage on the calibration distribution.

    alpha=0.05 -> target 95% coverage.
    """
    model.eval()
    scores = []

    for Xb, yb in calib_loader:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        mu, logvar = model(Xb)
        sigma = torch.exp(0.5 * logvar)

        s = (yb - mu).abs() / (sigma + eps)   # normalized residuals
        scores.append(s.squeeze(1).detach().cpu())

    scores = torch.cat(scores, dim=0)  # shape (N_cal,)

    # Conformal quantile (slightly conservative finite-sample choice):
    # k = ceil((n+1)*(1-alpha))
    n = scores.numel()
    k = int(torch.ceil(torch.tensor((n + 1) * (1 - alpha))).item())
    k = min(max(k, 1), n)

    q_hat = torch.kthvalue(scores, k).values.item()
    return q_hat

In [ ]:
# @torch.no_grad()
# def predict_with_conformal_interval(model, X, device, q_hat, eps=1e-6):
#     model.eval()
#     X = X.to(device, non_blocking=True)
#     mu, logvar = model(X)
#     sigma = torch.exp(0.5 * logvar)

#     lo = mu - q_hat * (sigma + eps)
#     hi = mu + q_hat * (sigma + eps)
#     return mu, sigma, lo, hi

In [ ]:
def train_one_model_hetero_with_conformal(model, train_loader, val_loader, calib_loader, model_name, lr=1e-4, epochs=80, 
                                          out_dir=None, alpha=0.05, eps=1e-6,):
    """
    Updates:
      1) Tracks sigma statistics (mean/min/max) for TRAIN and VAL each epoch.
         sigma = exp(0.5 * logvar)
      2) Prints those sigma stats in the epoch log line.

    Notes:
      - This does NOT change training behavior, it only adds diagnostics.
      - eps is only used for numeric stability if you later want it.
    """

    log_path = os.path.join(out_dir, f"{model_name}_training.log")

    logger = logging.getLogger(model_name)
    logger.setLevel(logging.INFO)

    # Remove old handlers if re-running in notebook
    if logger.hasHandlers():
        logger.handlers.clear()

    fh = logging.FileHandler(log_path)
    fh.setLevel(logging.INFO)

    formatter = logging.Formatter("%(message)s")
    fh.setFormatter(formatter)

    logger.addHandler(fh)

    if out_dir is None:
        out_dir = "."
    os.makedirs(out_dir, exist_ok=True)

    print("Model device: ", device)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []
    train_rmses, train_maes = [], []
    val_rmses, val_maes = [], []

    # NEW: store sigma stats over epochs (optional, but useful for plots)
    train_sigma_mean, train_sigma_min, train_sigma_max = [], [], []
    val_sigma_mean, val_sigma_min, val_sigma_max = [], [], []

    best_val = float("inf")
    best_path = os.path.join(out_dir, f"{model_name}_best.pt")

    for epoch in range(1, epochs + 1):

        # -------- Train --------
        model.train()
        running, n = 0.0, 0
        tr_mu_list, tr_true_list = [], []

        # NEW: sigma trackers
        tr_sigma_sum = 0.0
        tr_sigma_min = float("inf")
        tr_sigma_max = -float("inf")

        for Xb, yb in train_loader:
            Xb = Xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            mu, logvar = model(Xb)

            # NEW: sigma stats (in same units as y)
            sigma = torch.exp(0.5 * logvar)
            tr_sigma_sum += sigma.sum().item()
            tr_sigma_min = min(tr_sigma_min, sigma.min().item())
            tr_sigma_max = max(tr_sigma_max, sigma.max().item())

            loss = heteroscedastic_gaussian_nll(mu, logvar, yb)
            loss.backward()
            optimizer.step()

            running += loss.item() * Xb.size(0)
            n += Xb.size(0)

            tr_mu_list.append(mu.detach())
            tr_true_list.append(yb.detach())

        train_loss = running / n
        train_losses.append(train_loss)

        y_pred_tr = torch.cat(tr_mu_list, dim=0)
        y_true_tr = torch.cat(tr_true_list, dim=0)
        rmse_tr, mae_tr = compute_metrics(y_pred_tr, y_true_tr)
        train_rmses.append(rmse_tr)
        train_maes.append(mae_tr)

        # NEW: finalize train sigma stats
        tr_sig_mean = tr_sigma_sum / n
        train_sigma_mean.append(tr_sig_mean)
        train_sigma_min.append(tr_sigma_min)
        train_sigma_max.append(tr_sigma_max)

        # -------- Val --------
        model.eval()
        running, n = 0.0, 0
        va_mu_list, va_true_list = [], []

        # NEW: sigma trackers
        va_sigma_sum = 0.0
        va_sigma_min = float("inf")
        va_sigma_max = -float("inf")

        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                mu, logvar = model(Xb)

                # NEW: sigma stats
                sigma = torch.exp(0.5 * logvar)
                va_sigma_sum += sigma.sum().item()
                va_sigma_min = min(va_sigma_min, sigma.min().item())
                va_sigma_max = max(va_sigma_max, sigma.max().item())

                loss = heteroscedastic_gaussian_nll(mu, logvar, yb)

                running += loss.item() * Xb.size(0)
                n += Xb.size(0)

                va_mu_list.append(mu)
                va_true_list.append(yb)

        val_loss = running / n
        val_losses.append(val_loss)

        y_pred_va = torch.cat(va_mu_list, dim=0)
        y_true_va = torch.cat(va_true_list, dim=0)
        rmse_va, mae_va = compute_metrics(y_pred_va, y_true_va)
        val_rmses.append(rmse_va)
        val_maes.append(mae_va)

        # NEW: finalize val sigma stats
        va_sig_mean = va_sigma_sum / n
        val_sigma_mean.append(va_sig_mean)
        val_sigma_min.append(va_sigma_min)
        val_sigma_max.append(va_sigma_max)

        # -------- Checkpoint --------
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)

        # NEW: print sigma stats too
        msg = f"[{model_name}] Epoch {epoch:03d}/{epochs} | Train NLL: {train_loss:.6f} | Val NLL: {val_loss:.6f} | Train RMSE: {rmse_tr:.6f} | Val RMSE: {rmse_va:.6f} | Train MAE: {mae_tr:.6f} | Val MAE: {mae_va:.6f} | Train sigma(mean/min/max): {tr_sig_mean:.4f}/{tr_sigma_min:.4f}/{tr_sigma_max:.4f} | Val sigma(mean/min/max): {va_sig_mean:.4f}/{va_sigma_min:.4f}/{va_sigma_max:.4f}"
        print(msg)        # still prints to console
        logger.info(msg)  # also saves to file

    # -------- Load best model and calibrate q --------
    model.load_state_dict(torch.load(best_path, map_location=device))
    q_hat = conformal_calibrate_q(model, calib_loader, device=device, alpha=alpha)

    print(f"Saved best model: {best_path}")

    cal_msg = f"Conformal calibration: alpha={alpha} -> q_hat={q_hat:.4f}"
    print(cal_msg)
    logger.info(cal_msg)

    # Optional plot (rename MSE->NLL)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

    axes[0].plot(train_losses, label="Train NLL")
    axes[0].plot(val_losses, label="Val NLL")
    axes[0].set_title("NLL (training objective)")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("NLL")
    axes[0].grid(color='gray', linestyle='--', linewidth=0.5)
    axes[0].legend()

    axes[1].plot(train_rmses, label="Train RMSE")
    axes[1].plot(val_rmses, label="Val RMSE")
    axes[1].set_title("RMSE")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("RMSE")
    axes[1].grid(color='gray', linestyle='--', linewidth=0.5)
    axes[1].legend()

    axes[2].plot(train_maes, label="Train MAE")
    axes[2].plot(val_maes, label="Val MAE")
    axes[2].set_title("MAE")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("MAE")
    axes[2].grid(color='gray', linestyle='--', linewidth=0.5)
    axes[2].legend()

    plt.tight_layout()
    plot_path = os.path.join(out_dir, f"{model_name}_metrics_nll_rmse_mae.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved metrics figure: {plot_path}")

    return {
        "train_nll": train_losses,
        "val_nll": val_losses,
        "train_rmse": train_rmses,
        "val_rmse": val_rmses,
        "train_mae": train_maes,
        "val_mae": val_maes,
        "train_sigma_mean": train_sigma_mean,
        "train_sigma_min": train_sigma_min,
        "train_sigma_max": train_sigma_max,
        "val_sigma_mean": val_sigma_mean,
        "val_sigma_min": val_sigma_min,
        "val_sigma_max": val_sigma_max,
        "best_model_path": best_path,
        "metrics_fig_path": plot_path,
        "q_hat": q_hat,
    }

In [ ]:
%%time

lstm_hetero_attn = LSTMAttnHeteroRegressor(n_features=8, hidden_dim=128, num_layers=2, dropout=0.1, bidirectional=False)
lstm_hist = train_one_model_hetero_with_conformal(lstm_hetero_attn, train_loader, val_loader,calib_loader,
                             model_name="lstm_hetero_attn", lr=1e-4, epochs=200, out_dir=f"{output_path}/{sim_num}")

In [ ]:
%%time

tcn_hetero_attn = TCN_Atten_HeteroRegressor(n_features=8, channels=(64, 64, 64), kernel_size=3, dropout=0.1)
tcn_hist = train_one_model_hetero_with_conformal(tcn_hetero_attn, train_loader, val_loader, calib_loader, model_name='tcn_hetero_attn', 
                                                 lr=1e-4, epochs=200, out_dir=f"{output_path}/{sim_num}")

# Evaluation

In [ ]:
import datetime

In [ ]:
required_yr = 2018
str(datetime.datetime.now())

In [ ]:
sim_num = 1
list_of_files = glob.glob(f"DATA-PATH")
output_path = '_files_for_analysis_'

In [ ]:
%%time

import re
# --- config ---
FEAT_COLS = ["SST","SAL","ice_frac","mixed_layer_depth","heat_flux_down",
    "water_flux_up","wind_stress_curl_f","rel_vorticity_f"]
TGT_COL = "co2flux_pre"
KEEP_COLS = ["nav_lat","nav_lon","time_counter"] + FEAT_COLS + [TGT_COL]


feature_dict = defaultdict(list)  # (lat, lon) -> list of (12, 8)
target_dict  = defaultdict(list)  # (lat, lon) -> list of (12, 1)
coord_counter = defaultdict(int)

year_to_file = {}
year_pat = re.compile(r"_(\d{4})_")
for fp in list_of_files:
    m = year_pat.search(fp)
    if m:
        yr = int(m.group(1))
        year_to_file.setdefault(yr, fp)

for yr in range(required_yr-1, required_yr+1):
    print(yr)
    fp = year_to_file.get(yr)
    if fp is None:
        raise FileNotFoundError(f"No file found for year {yr}")

    df = pd.read_pickle(fp)

    # mask land
    df = df[df["tmask"] == 1]
    df = util_functions.round_nav_lat(df)

  
    df = df[KEEP_COLS]

    # aggregate once: (lat, lon, month) -> mean
    df = (
        df.groupby(["nav_lat","nav_lon","time_counter"], sort=False, observed=True)
          .mean(numeric_only=True)
          .reset_index()
    )

    df = df.sort_values(["nav_lat","nav_lon","time_counter"], kind="mergesort", ignore_index=True)

    counts = df.groupby(["nav_lat","nav_lon"], sort=False)["time_counter"].size().to_numpy()
    if np.any(counts != 12):
        bad = df.groupby(["nav_lat","nav_lon"], sort=False)["time_counter"].size()
        bad = bad[bad != 12]
        raise ValueError(f"Found coords with != 12 months in year {yr}. Examples:\n{bad.head()}")

    n_cells = counts.size
    X = df[FEAT_COLS].to_numpy(dtype=np.float32, copy=False).reshape(n_cells, 12, len(FEAT_COLS))
    y = df[[TGT_COL]].to_numpy(dtype=np.float32, copy=False).reshape(n_cells, 12, 1)

    keys = df[["nav_lat","nav_lon"]].to_numpy(copy=False)[::12]

    # append into dicts (still a loop, but now over n_cells once/year, not over group objects)
    for (lat, lon), Xi, yi in zip(map(tuple, keys), X, y):
        feature_dict[(lat, lon)].append(Xi)
        target_dict[(lat, lon)].append(yi)
        coord_counter[(lat, lon)] += 1

In [ ]:
%%time

from numpy.lib.stride_tricks import sliding_window_view

def build_xy_from_dicts(feature_dict, target_dict, lookback=6, dtype=np.float32, start_year=None):
    """
    Returns:
      X_all:    (N_samples, lookback, 8)
      y_all:    (N_samples, 1)
      coord_all:(N_samples, 2)   lat, lon
      time_all: (N_samples, 2)   year, month
    """
    X_list, y_list, coord_list, time_list = [], [], [], []

    for (lat, lon), year_blocks in feature_dict.items():
        if (lat, lon) not in target_dict:
            continue

        # (T, 8)
        X_ts = np.concatenate(year_blocks, axis=0).astype(dtype, copy=False)
        # (T, 1)
        y_ts = np.concatenate(target_dict[(lat, lon)], axis=0).astype(dtype, copy=False)

        T, F = X_ts.shape
        if F != 8:
            raise ValueError(f"Expected 8 features, got {F}")
        if y_ts.shape != (T, 1):
            raise ValueError(f"Target shape mismatch: {y_ts.shape} vs {(T,1)}")

        # pad for early months
        pad = np.zeros((lookback - 1, F), dtype=dtype)
        X_pad = np.vstack([pad, X_ts])  # (T+lookback-1, 8)  (allocates)

        # windows as a view (no copy)
        X_win = sliding_window_view(X_pad, window_shape=(lookback,), axis=0)  # (T, 8, lookback)
        X_win = np.transpose(X_win, (0, 2, 1))  # (T, lookback, 8)

        y_win = y_ts  # (T, 1)

        X_list.append(X_win)
        y_list.append(y_win)

        # coords (T,2)
        coord_list.append(np.tile(np.array([[lat, lon]], dtype=dtype), (T, 1)))

        # time (T,2): year, month
        # t=0 is Jan of start_year
        t = np.arange(T, dtype=np.int32)
        years = start_year + (t // 12)
        months = 1 + (t % 12)
        time_list.append(np.stack([years, months], axis=1).astype(np.int16, copy=False))

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    coord_all = np.concatenate(coord_list, axis=0)
    time_all = np.concatenate(time_list, axis=0)

    return X_all, y_all, coord_all, time_all


X_test_np, y_test_np, coord_test_np, time_test_np = build_xy_from_dicts(feature_dict, target_dict, lookback=6, start_year=required_yr-1)

print(X_test_np.shape)
print(y_test_np.shape)
print(coord_test_np.shape)
print(time_test_np.shape)  # (N_samples, 2) year, month

In [ ]:
coord_test_np[0]

In [ ]:
test_data = np.hstack([coord_test_np, time_test_np, y_test_np])
meta_test_df = pd.DataFrame(test_data,columns=["nav_lat", "nav_lon", "year", "month", "co2flux_pre_simulated"])
meta_test_df

In [ ]:
valid = np.isfinite(X_test_np).all(axis=(1,2)) & np.isfinite(y_test_np).all(axis=1)
print("Removed samples:", (~valid).sum())
X_test_np = X_test_np[valid]
y_test_np = y_test_np[valid]
meta_test_df = meta_test_df.loc[valid].reset_index(drop=True)
meta_test_df

In [ ]:
x_scaler = joblib.load(f"{output_path}/{sim_num}/x_scaler_k_2000000_for_cal.joblib")

In [ ]:
%%time
# Scale X_test_win: scaler expects 2D (N*window, F)
Ns, W, F = X_test_np.shape
X_test_reshape = X_test_np.reshape(-1, F)
X_test_np_s = x_scaler.transform(X_test_reshape).reshape(Ns, W, F).astype(np.float32)

X_t = torch.from_numpy(X_test_np_s)
y_t = torch.from_numpy(y_test_np)

test_loader = DataLoader(TensorDataset(X_t, y_t), batch_size=2048, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


# model_path = f"{output_path}/{sim_num}/tcn_hetero_attn_best.pt"
model_path = f"{output_path}/{sim_num}/lstm_hetero_attn_best.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

select_model = LSTMAttnHeteroRegressor(n_features=8, hidden_dim=128, num_layers=2, dropout=0.1, bidirectional=False)
# select_model = TCNAttenHeteroRegressor(n_features=8, channels=(64, 64, 64), kernel_size=3, dropout=0.1, return_attention=True, )

state = torch.load(model_path, map_location="cpu")   # safest across nodes
select_model.load_state_dict(state)

select_model = select_model.to(device)          
select_model.eval()

In [ ]:
%%time

batch_size = 2048
test_loader = DataLoader(TensorDataset(X_t), batch_size=batch_size,
                         shuffle=False, num_workers=0, pin_memory=True)

mu_list, logvar_list, attn_list = [], [], []

with torch.no_grad():
    for (Xb,) in test_loader:
        Xb = Xb.to(device, non_blocking=True)

        mu_b, logvar_b, attn_b = select_model(Xb, return_attn=True)  # mu:(B,1), logvar:(B,1), attn:(B,T)

        mu_list.append(mu_b.cpu().numpy())
        logvar_list.append(logvar_b.cpu().numpy())
        attn_list.append(attn_b.cpu().numpy())

mu     = np.vstack(mu_list)        # (Ns, 1)
logvar = np.vstack(logvar_list)    # (Ns, 1)
attn_w = np.vstack(attn_list)      # (Ns, W)

# Derived uncertainty (optional but usually what you want)
var   = np.exp(logvar)             # (Ns, 1)
sigma = np.sqrt(var)               # (Ns, 1)

In [ ]:
y_true = y_test_np.reshape(-1, 1).astype(np.float32)

rmse = float(np.sqrt(np.mean((mu - y_true) ** 2)))
mae  = float(np.mean(np.abs(mu - y_true)))

print(f"TEST RMSE: {rmse:.6f}")
print(f"TEST MAE:  {mae:.6f}")

In [ ]:
attn_w.shape

In [ ]:
pred_df = meta_test_df.copy()

pred_df["co2flux_pre_simulated"] = y_true.reshape(-1)
pred_df["co2flux_pre_reconstructed"] = mu.reshape(-1)
pred_df["co2flux_pre_reconstructed_log_var"] = logvar.reshape(-1)

## full vectors as lists
# pred_df["attention_weights"] = attn_w.tolist()   # each row: length T
# pred_df["context_vector"]    = context.tolist()  # each row: length H

T = attn_w.shape[1]
# H = context.shape[1]

for t in range(T):
    # pred_df[f"attn_lag_{t}"] = attn_w[:, t]

    lag = T - 1 - t     # maps 0→5, 5→0
    pred_df[f"attn_lag_{lag}"] = attn_w[:, t]

# for h in range(H):
#     pred_df[f"context_{h}"] = context[:, h]

In [ ]:
assert len(pred_df) == attn_w.shape[0]

In [ ]:
pred_df = pred_df.loc[pred_df['year'] == required_yr]
pred_df

In [ ]:
%%time
save_path = f"{output_path}/{sim_num}/19_test_years/lstm_attention_with_calib/lstm_attention_with_calib_predictions_{required_yr}.parquet"
# save_path = f"{output_path}/{sim_num}/19_test_years/tcn_attention_with_calib/tcn_attention_with_calib_predictions_{required_yr}.parquet"
pred_df.to_parquet(save_path, index=False)
print("Saved:", save_path)
str(datetime.datetime.now())